# Wordprint on Colab

用 Google 的 GPU 跑 Qwen 轉錄並分說話者，不必在自己電腦安裝模型，也不需要 Hugging Face 帳號或 token。音檔會上傳到這個 Colab 執行環境（Google 的機器）。

1. 「執行階段 → 全部執行」，在第 3 格選音檔上傳。不用重新啟動。
2. 最後會下載 `normalized-run-*.json`；交給 Wordprint 繼續（驗證、閱讀頁、摘要、郵件）。

預設用 T4 GPU；沒分配到 GPU 時照樣用 CPU 跑，並自動改用較快的 0.6B 模型。說話者由 NVIDIA Nemotron-3-Diarization 標記（最多 8 人）。

不在 Colab（本機 Jupyter，未實測）：第 1 格填 `AUDIO_FILES`，結果寫在目前資料夾。

In [ ]:
MODEL_SIZE = "1.7B"  # 或 "0.6B"（較快、堪用）；沒有 GPU 時自動改用 0.6B
MERGE = False  # 一次上傳多個檔案時，True＝依上傳順序合併成一段；False＝每個檔案各自一份結果
MAX_NEW_TOKENS = 2048  # 每段（≤3 分鐘）輸出上限；碰到上限的結果會標成 truncated
AUDIO_FILES = []  # 只在本機 Jupyter 用：音檔路徑（Colab 改用上傳）

In [ ]:
# 一次解析全部版本：qwen-asr 鎖定 transformers 4.57.6，Nemotron 需要 NeMo 的這個版本。
# pip 可能印出 google-adk、diffusers 等 Colab 內建套件的相依衝突 ERROR；本筆記本用不到它們，可以忽略。
!pip install -q qwen-asr transformers==4.57.6 Cython packaging "nemo-toolkit[asr] @ https://github.com/NVIDIA/NeMo/archive/16b0ff2584c6355c53cbd6c28f0f431a76b24956.zip"

In [ ]:
import shlex, sys
from pathlib import Path
try:
    from google.colab import files
    names = list(files.upload())  # 檔案存到目前目錄
except ImportError:  # 本機 Jupyter
    files, names = None, AUDIO_FILES
groups = [names] if MERGE else [[name] for name in names]
print("將轉錄：", groups)

In [ ]:
%%writefile torch_worker.py
#!/usr/bin/env python3
"""PyTorch worker for machines without the MLX worker (Windows, Linux, Intel Mac) and for the Colab notebook.

Qwen3-ASR + ForcedAligner via qwen-asr, speakers via NVIDIA Nemotron-3-Diarization through NeMo (no Hugging Face
token). Writes one normalized-run.json (references/run-format.md). The Colab notebook embeds this file verbatim
(tests/test_notebook.py keeps them in sync). Runs in its own process so the qwen-asr pin on transformers never meets
another version already loaded in a notebook session.
"""
import argparse
import hashlib
import importlib.metadata
import json
import subprocess
import sys
import unicodedata
import wave
from pathlib import Path

ALIGNER = "Qwen/Qwen3-ForcedAligner-0.6B"
DIARIZER = "nvidia/Nemotron-3-Diarization"  # OpenMDW-1.1, not gated; up to 8 speakers.


def decode(name):
    """原始音檔 → 單聲道 16 kHz s16 PCM 位元組。"""
    return subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-i", name, "-ac", "1", "-ar", "16000", "-f", "s16le", "-"],
                          check=True, capture_output=True).stdout


def gap(text):
    """對時器略過的空白、標點、符號原樣併入鄰詞（時間不變）；略過真的字則不併入、不借鄰詞時間，讓 validate 擋下。"""
    if any(not c.isspace() and unicodedata.category(c)[0] not in "PS" for c in text):  # transcript.py normalized，另略符號
        print(f"警告：對時漏掉「{text}」；這份結果驗證不會通過", flush=True)
        return ""
    return text


def transcribe(asr, names, model, max_new_tokens, platform, dtype):
    """qwen-asr 自己把長音訊切成 ≤180 秒的段、逐段對時並換算成全段時間；這裡只把結果寫成 normalized run。"""
    import numpy as np
    generated = []  # transcribe() 不回傳停止原因；記下每批實際生成長度，碰到上限就標 truncated
    original = asr.model.generate

    def generate(*args, **kwargs):
        output = original(*args, **kwargs)
        generated.append(output.sequences.shape[1] - kwargs["input_ids"].shape[1])
        return output

    asr.model.generate = generate
    pcm, sources = b"", []
    for name in names:
        data = decode(name)
        sources.append({"path": name, "sha256": hashlib.sha256(Path(name).read_bytes()).hexdigest(), "duration": len(data) / 32000,
                        "offset": len(pcm) / 32000, "sample_count": len(data) // 2, "sample_rate": 16000})
        pcm += data
    samples = np.frombuffer(pcm, dtype=np.int16).astype(np.float32) / 32768
    result = asr.transcribe((samples, 16000), return_time_stamps=True)[0]
    words, cursor = [], 0
    for item in result.time_stamps or []:
        found = result.text.find(item.text, cursor)
        if found < 0:  # 對時文字與原文不符：保留原樣，讓 validate 如實擋下
            words.append({"text": item.text, "start": item.start_time, "end": item.end_time})
            continue
        words.append({"text": gap(result.text[cursor:found]) + item.text, "start": item.start_time, "end": item.end_time})
        cursor = found + len(item.text)
    if words:
        words[-1]["text"] += gap(result.text[cursor:])
    duration = len(samples) / 16000
    run = {"schema_version": 1, "sources": sources, "platform": platform,
           "model": {"id": model, "size": model.rsplit("-", 1)[1], "backend": "qwen-asr " + importlib.metadata.version("qwen-asr"),
                     "language": result.language, "dtype": str(dtype), "max_new_tokens": max_new_tokens},
           "alignment_model": {"id": ALIGNER},
           "chunks": [{"start": 0, "end": duration, "text": result.text, "words": words, "complete": True,
                       "truncated": max(generated, default=0) >= max_new_tokens, "generated_tokens": list(generated)}],
           "turns": [], "diarization_complete": False, "speakers": {}}
    return run, samples, pcm


def diarize(run, samples, device):
    """Nemotron 一次處理整段（內部串流切塊），說話者依首次發言編成 S1、S2…；失敗時保留無說話者的結果並說明原因。"""
    import torch
    try:
        from nemo.collections.asr.models import SortformerEncLabelModel
        model = SortformerEncLabelModel.from_pretrained(DIARIZER, map_location=device).to(torch.float32).eval()
        modules = model.sortformer_modules  # 模型卡的離線設定
        modules.chunk_len, modules.chunk_right_context, modules.fifo_len, modules.spkcache_update_period = 340, 40, 40, 300
        model._check_streaming_parameters()
        segments = model.diarize(audio=[samples], batch_size=1, sample_rate=16000)[0]
    except Exception as error:
        print(f"這次不分說話者（{type(error).__name__}：{error}）。", flush=True)
        return
    order, duration, turns = {}, len(samples) / 16000, []
    for segment in sorted(segments, key=lambda s: float(s.split()[0])):
        start, end, label = segment.split()
        start, end = float(start), min(float(end), duration)
        if end > start:
            turns.append({"start": start, "end": end, "speaker": order.setdefault(label, f"S{len(order) + 1}")})
    run.update(turns=turns, speakers={speaker: {} for speaker in order.values()}, diarization_complete=True,
               diarization_model={"id": DIARIZER, "backend": "nemo-toolkit " + importlib.metadata.version("nemo-toolkit")})


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--audio", required=True, action="append", help="repeat to merge recordings in this order")
    parser.add_argument("--out", required=True, help="normalized-run.json to write")
    parser.add_argument("--model", default="0.6B", choices=["0.6B", "1.7B"])
    parser.add_argument("--max-new-tokens", type=int, default=2048, help="per ≤3-minute segment; hitting it marks truncated")
    parser.add_argument("--platform", default=sys.platform)
    parser.add_argument("--pcm", help="also write the canonical 16 kHz WAV here (sets run.pcm)")
    parser.add_argument("--cpu-falls-back-to-0.6b", dest="fallback", action="store_true",
                        help="notebook default: without a GPU run 0.6B instead of 1.7B and say so")
    args = parser.parse_args()
    import torch
    from qwen_asr import Qwen3ASRModel
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    if args.fallback and device == "cpu" and args.model == "1.7B":
        args.model = "0.6B"
        print("沒有分配到 GPU：改用 0.6B（CPU 上 1.7B 太慢）；要 1.7B 請換成 GPU 執行階段後重跑。", flush=True)
    model, dtype = f"Qwen/Qwen3-ASR-{args.model}", torch.float16 if device != "cpu" else torch.float32  # T4 沒有 bf16
    asr = Qwen3ASRModel.from_pretrained(model, dtype=dtype, device_map=device, max_new_tokens=args.max_new_tokens,
                                        forced_aligner=ALIGNER, forced_aligner_kwargs={"dtype": dtype, "device_map": device})
    print("轉錄中…", flush=True)
    run, samples, pcm = transcribe(asr, args.audio, model, args.max_new_tokens, f"{args.platform} {device}", dtype)
    print("分說話者中…", flush=True)
    diarize(run, samples, device)
    if args.pcm:
        with wave.open(args.pcm, "wb") as out:
            out.setnchannels(1), out.setsampwidth(2), out.setframerate(16000), out.writeframes(pcm)
        run["pcm"] = str(Path(args.pcm).resolve())
    Path(args.out).write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    print(args.out, "：", len(run["chunks"][0]["words"]), "詞，說話者", len(run["speakers"]) or "未標記", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
for names in groups:
    out = f"normalized-run-{Path(names[0]).stem}.json"
    Path(out).unlink(missing_ok=True)
    command = [sys.executable, "torch_worker.py", "--model", MODEL_SIZE, "--max-new-tokens", str(MAX_NEW_TOKENS),
               "--platform", "Google Colab" if files else "Jupyter", "--cpu-falls-back-to-0.6b", "--out", out]
    get_ipython().system(shlex.join(command + [arg for name in names for arg in ("--audio", name)]))
    if not Path(out).exists():
        raise RuntimeError(f"{names} 轉錄失敗，原因在上方訊息")
    if files:
        files.download(out)